# Instalacija biblioteka

In [ ]:
# ============================================================================
# Brain Tumor MRI Classifier - Google Colab GUI
# Tim Pikseli - Elektrotehnički fakultet Sarajevo 2026
# ============================================================================

print("📦 Instalacija potrebnih biblioteka...")
!pip install -q gradio tensorflow opencv-python-headless pillow

print("✅ Instalacija završena!")

📦 Instalacija potrebnih biblioteka...
✅ Instalacija završena!


# Mount Google Drive i pronalaženje modela

In [ ]:
from google.colab import drive
import os

print("📁 Montiranje Google Drive-a...")
drive.mount('/content/drive')

# Moguce putanje do modela
POSSIBLE_PATHS = [
    '/content/drive/Shareddrives/POOS projekat/models/best_brain_tumor_model.keras',
    '/content/drive/.shortcut-targets-by-id/*/models/best_brain_tumor_model.keras',
    '/content/drive/MyDrive/POOS projekat/models/best_brain_tumor_model.keras'
]

MODEL_PATH = None

# Provjera Shared Drive
shared_path = '/content/drive/Shareddrives/POOS projekat/models/best_brain_tumor_model.keras'
if os.path.exists(shared_path):
    MODEL_PATH = shared_path
    print(f"✅ Model pronađen u Shared Drive: {MODEL_PATH}")
else:
    # Provjera shortcut foldera (za "Shared with me")
    import glob
    shortcut_search = '/content/drive/.shortcut-targets-by-id/*/POOS projekat/models/best_brain_tumor_model.keras'
    matches = glob.glob(shortcut_search)

    if matches:
        MODEL_PATH = matches[0]
        print(f"✅ Model pronađen u 'Shared with me': {MODEL_PATH}")
    else:
        # Provjera MyDrive
        mydrive_path = '/content/drive/MyDrive/POOS projekat/models/best_brain_tumor_model.keras'
        if os.path.exists(mydrive_path):
            MODEL_PATH = mydrive_path
            print(f"✅ Model pronađen u MyDrive: {MODEL_PATH}")
        else:
            print("❌ GREŠKA: Model nije pronađen ni na jednoj lokaciji!")
            print("\n📍 Molimo provjeri sljedeće:")
            print("1. Da li je 'POOS projekat' folder dijeljenja (Shared)?")
            print("2. Da li si kliknuo na link i dao pristup Drive-u?")
            print("3. Da li folder 'models' postoji?")
            print("\n🔍 Pokušaj ručno:")
            print("   - Otvori Google Drive")
            print("   - Desni klik na 'POOS projekat' folder")
            print("   - Odaberi 'Add shortcut to Drive'")
            print("   - Stavi ga u 'My Drive'")
            print("   - Ponovo pokreni ovaj cell")


📁 Montiranje Google Drive-a...
Mounted at /content/drive
✅ Model pronađen u MyDrive: /content/drive/MyDrive/POOS projekat/models/best_brain_tumor_model.keras


# Import biblioteka

In [ ]:
import gradio as gr
import tensorflow as tf
import cv2
import numpy as np
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print("✅ Biblioteke uspješno importovane!")

✅ Biblioteke uspješno importovane!


# Definicije klasa i opisa

In [ ]:
CLASS_NAMES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']

TUMOR_INFO = {
    'Glioma': {
        'emoji': '🔴',
        'color': '#D32F2F',
        'desc': 'Tumor glija ćelija mozga. Potrebna je hitna medicinska evaluacija i konsultacija sa neurohirurgom.'
    },
    'Meningioma': {
        'emoji': '🟠',
        'color': '#F57C00',
        'desc': 'Tumor moždanih ovojnica. Obično benigan, ali zahtijeva redovno praćenje i medicinsku pažnju.'
    },
    'Pituitary': {
        'emoji': '🟡',
        'color': '#FFA726',
        'desc': 'Tumor hipofize. Može uticati na hormonalnu ravnotežu i zahtijeva specijalističko endokrinološko liječenje.'
    },
    'No Tumor': {
        'emoji': '🟢',
        'color': '#388E3C',
        'desc': 'Na snimku nisu detektovane abnormalnosti. Moždano tkivo izgleda uredno i zdravo.'
    }
}

print("✅ Konfiguracija učitana!")

✅ Konfiguracija učitana!


# Učitavanje modela

In [ ]:
print("🤖 Učitavanje CNN modela...")

try:
    model = tf.keras.models.load_model(MODEL_PATH)
    print("✅ Model uspješno učitan!")
    print(f"   - Ulazna dimenzija: {model.input_shape}")
    print(f"   - Broj klasa: {len(CLASS_NAMES)}")
except Exception as e:
    print(f"❌ GREŠKA pri učitavanju modela: {e}")
    model = None

🤖 Učitavanje CNN modela...
✅ Model uspješno učitan!
   - Ulazna dimenzija: (None, 224, 224, 3)
   - Broj klasa: 4


# Funkcija za preprocecsiranje

In [ ]:
def preprocess_mri_image(img):
    """
    Preprocesira MRI sliku prema pipeline-u korištenom u treningu:
    1. Gaussian blur-based sharpening
    2. CLAHE enhancement
    3. Resize na 224x224
    4. Normalizacija [0, 1]
    """
    try:
        # Konverzija PIL Image u numpy array
        img_array = np.array(img)

        # Obrada različitih formata
        if len(img_array.shape) == 2:
            img_rgb = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)
        elif img_array.shape[2] == 4:
            img_rgb = cv2.cvtColor(img_array, cv2.COLOR_RGBA2RGB)
        else:
            img_rgb = img_array

        # Korak 1: Sharpening sa Gaussian blur
        blurred = cv2.GaussianBlur(img_rgb, (5, 5), 0)
        sharpened = cv2.addWeighted(img_rgb, 1.5, blurred, -0.5, 0)

        # Korak 2: Konverzija u grayscale
        gray = cv2.cvtColor(sharpened, cv2.COLOR_RGB2GRAY)

        # Korak 3: CLAHE enhancement
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        clahe_img = clahe.apply(gray)

        # Korak 4: Konverzija nazad u RGB
        clahe_rgb = cv2.cvtColor(clahe_img, cv2.COLOR_GRAY2RGB)

        # Korak 5: Resize na 224x224
        resized = cv2.resize(clahe_rgb, (224, 224), interpolation=cv2.INTER_AREA)

        # Korak 6: Normalizacija
        normalized = resized.astype('float32') / 255.0

        return np.expand_dims(normalized, axis=0)

    except Exception as e:
        print(f"Greška u preprocesiranju: {e}")
        return None

print("✅ Funkcija za preprocesiranje definisana!")

✅ Funkcija za preprocesiranje definisana!


# Funkcija za klasifikaciju

In [ ]:
def classify_brain_tumor(image):
    """
    Klasifikuje MRI sliku i vraća detaljne rezultate
    """
    if image is None:
        return (
            "⚠️ Molimo učitajte MRI sliku.",
            {},
            "Nema slike za analizu."
        )

    if model is None:
        return (
            "❌ Model nije učitan.",
            {},
            "Greška: Model nije dostupan."
        )

    # Preprocesiranje slike
    processed = preprocess_mri_image(image)

    if processed is None:
        return (
            "❌ Greška pri obradi slike.",
            {},
            "Slika se nije mogla preprocesirati."
        )

    # Predikcija
    predictions = model.predict(processed, verbose=0)

    # Rezultati
    class_idx = np.argmax(predictions[0])
    confidence = np.max(predictions[0])
    predicted_class = CLASS_NAMES[class_idx]

    # Priprema rezultata
    info = TUMOR_INFO[predicted_class]

    # Formatiran tekst rezultata
    result_text = f"""
## {info['emoji']} Detektovana Klasa: **{predicted_class}**

### 📊 Pouzdanost: **{confidence:.1%}**

### 📋 Medicinske Napomene:
{info['desc']}

---
⚠️ **NAPOMENA:** Ova dijagnoza je generisana AI modelom (Accuracy: 91.3%) i ne zamjenjuje profesionalnu medicinsku konsultaciju.
"""

    # Dictionary sa svim pouzdanostima
    confidences_dict = {
        CLASS_NAMES[i]: float(predictions[0][i])
        for i in range(4)
    }

    # Dodatne informacije
    additional_info = f"{info['emoji']} {info['desc']}"

    return result_text, confidences_dict, additional_info

print("✅ Funkcija za klasifikaciju definisana!")

✅ Funkcija za klasifikaciju definisana!


# Kreiranje Gradio interfejsa

In [ ]:
print("🎨 Kreiranje Gradio interfejsa...")

# Custom CSS za bolji izgled
custom_css = """
.gradio-container {
    font-family: 'Segoe UI', sans-serif;
}
.header {
    text-align: center;
    padding: 20px;
    background: linear-gradient(135deg, #1565C0 0%, #1976D2 100%);
    color: white;
    border-radius: 10px;
    margin-bottom: 20px;
}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:

    # Header
    gr.Markdown(
        """
        <div class="header">
            <h1>🧠 Brain Tumor MRI Classifier</h1>
            <h3>Automatska klasifikacija MRI snimaka pomoću CNN modela</h3>
            <p><b>Tim Pikseli</b> | Elektrotehnički fakultet Sarajevo | 2026</p>
        </div>
        """
    )

    with gr.Row():
        # Lijevi panel - Upload slike
        with gr.Column(scale=1):
            gr.Markdown("### 📊 Upload MRI Snimka")

            image_input = gr.Image(
                type="pil",
                label="Učitajte MRI sliku",
                height=400
            )

            with gr.Row():
                clear_btn = gr.Button("🗑️ Očisti", variant="secondary")
                analyze_btn = gr.Button("🔍 Analiziraj", variant="primary", scale=2)

            gr.Markdown(
                """
                **Podržani formati:** JPG, PNG, JPEG
                **Preporučena rezolucija:** 224x224 ili veća

                **Klase tumora:**
                - 🔴 Glioma
                - 🟠 Meningioma
                - 🟡 Pituitary
                - 🟢 No Tumor
                """
            )

        # Desni panel - Rezultati
        with gr.Column(scale=1):
            gr.Markdown("### 📋 Dijagnostički Izvještaj")

            result_output = gr.Markdown(
                value="*Čekanje na analizu...*\n\nUčitajte MRI sliku i kliknite na dugme 'Analiziraj'."
            )

            gr.Markdown("### 📈 Pouzdanost Po Klasama")

            confidence_output = gr.Label(
                label="Distribucija vjerovatnoća",
                num_top_classes=4
            )

            gr.Markdown("### 💡 Dodatne Informacije")

            info_output = gr.Textbox(
                label="Medicinske napomene",
                lines=4,
                interactive=False
            )

    # Footer
    gr.Markdown(
        """
        ---
        ### 👥 Tim Pikseli
        **Studenti:** Ali Ljaljak, Daris Mujkić, Amina Kazazović, Benjamin Hadžihasanović, Elma Tirak

        **Projekat:** Prepoznavanje oblika i obrada slike
        **Institucija:** Univerzitet u Sarajevu - Elektrotehnički fakultet

        **Model Accuracy:** 91.30% | **Dataset:** Brain Tumor MRI Dataset (7023 slike)

        ---
        © 2026 Tim Pikseli - Sva prava zadržana
        """
    )

    # Event handlers
    analyze_btn.click(
        fn=classify_brain_tumor,
        inputs=[image_input],
        outputs=[result_output, confidence_output, info_output]
    )

    clear_btn.click(
        fn=lambda: (None, "*Čekanje na analizu...*", {}, ""),
        inputs=[],
        outputs=[image_input, result_output, confidence_output, info_output]
    )

print("✅ Interfejs kreiran!")

🎨 Kreiranje Gradio interfejsa...
✅ Interfejs kreiran!


# Pokretanje aplikacije

In [ ]:
print("\n" + "="*60)
print("🚀 POKRETANJE APLIKACIJE")
print("="*60)

demo.launch(
    share=True,           # Kreira javni link
    debug=True,           # Debug mod
    show_error=True,      # Prikazuje greške
    inline=False          # Otvara u novom tabu
)

print("\n✅ Aplikacija je pokrenuta!")
print("📱 Kliknite na javni link (share link) da pristupite interfejsu")
print("🔗 Link možete podijeliti sa bilo kim!")


🚀 POKRETANJE APLIKACIJE
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f4b12bdc0134a25ca4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
